## Silver Layer — ERCOT Wind & Solar Generation

Reads from Bronze, applies all data quality transformations, and writes
a clean conformed table with one row per UTC hour.

**Source:** workspace.medallion_lakehouse.bronze_generation. 

**Target:** workspace.medallion_lakehouse.silver_generation. 

**Rows:** 26,298 (one per UTC hour, 2023-2025)

**Transformations applied:**
- Renamed all columns to snake_case (Bronze)
- Parsed hour_ending string → proper timestamp
- Extracted year, month, day, hour_of_day from timestamp
- Split wind and solar rows (originally stacked from separate Excel tabs)
- Fixed solar nighttime values: 1 → 0 where is_daytime_hour = False
- Dropped 96 empty rows where both wind and solar were NULL
- Converted local Central Time → UTC using month-based offset logic
- Resolved DST duplicate timestamps via deduplication (keep higher generation)
- Joined wind and solar into one row per UTC timestamp

**Known limitations:**
- Source data (ERCOT public file) does not include per-row timezone indicator
- DST transition hours (3 per year) use month-based UTC offset approximation
- 6 rows dropped during DST deduplication — one observation retained per hour
- timestamp_utc is the canonical timestamp — original CST timestamp dropped

In [0]:
from pyspark.sql.functions import (to_utc_timestamp,
    to_timestamp, year, month, dayofmonth, hour, when, col, lit, count, sum as spark_sum, row_number, expr
)
from pyspark.sql.window import Window
import pyspark.sql.functions as F

In [0]:
bronze_clean_df = spark.table("workspace.medallion_lakehouse.bronze_generation")
print(f"Bronze rows: {bronze_clean_df.count()}")
bronze_clean_df.printSchema()

In [0]:
# Parse the timestamp string into a real timestamp and extract useful date components as separate columns
silver_df = bronze_clean_df \
    .withColumn("timestamp", to_timestamp(col("hour_ending"), "MM/dd/yyyy HH:mm:ss")) \
    .withColumn("year", year(col("timestamp"))) \
    .withColumn("month", month(col("timestamp"))) \
    .withColumn("day", dayofmonth(col("timestamp"))) \
    .withColumn("hour_of_day", hour(col("timestamp")))

# Verify timestamp parsed correctly
silver_df.select("hour_ending","timestamp", "year","month","day","hour_of_day").show(5)

In [0]:
# Fix solar nighttime values and drop unused columns
silver_df = silver_df \
    .withColumn("solar_gen_mw", when((col("solar_gen_mw") == 1) & (col("daytime_hour") == False),
                lit(0)).otherwise(col("solar_gen_mw"))).drop("date_label","source_table")
    
# verify the fix worked
silver_df.select("timestamp","hour_of_day","daytime_hour","solar_gen_mw").show(10)

In [0]:
# How many rows have wind data vs solar data?
silver_df.select(
    count("*").alias("total_rows"),
    spark_sum(
        when(col("wind_gen_mw").isNotNull(),1).otherwise(0)).alias("wind_rows"),
    spark_sum(
        when(col("solar_gen_mw").isNotNull(),1).otherwise(0)).alias("solar_rows"),
    spark_sum(
        when(col("wind_gen_mw").isNotNull() & col("solar_gen_mw").isNotNull(),1).otherwise(0)).alias("both_rows_present")
    ).show()

In [0]:
# Find the missing 96 rows
mystery_df = silver_df.filter(
    col("wind_gen_mw").isNull() & col("solar_gen_mw").isNull()
)
print(f"Mystery rows: {mystery_df.count()}")
mystery_df.select(
    "timestamp","hour_ending","ercot_load_mw","wind_gen_mw","solar_gen_mw","daytime_hour"
).show()

In [0]:
# Drop rows where both wind and solar are null
# These are empty/header rows with no useful data
# This filter is robust to catch the same issue in future uploads

rows_before = silver_df.count()

silver_df = silver_df.filter(
    col("wind_gen_mw").isNotNull() | col("solar_gen_mw").isNotNull()
)

rows_after = silver_df.count()
rows_dropped = rows_before - rows_after

print(f"Rows before: {rows_before}")
print(f"Rows after: {rows_after}")
print(f"Rows dropped: {rows_dropped}")

if rows_dropped > 0:
    print(f"WARNING: {rows_dropped} empty rows detected and removed")
else:
    print("No empty rows found - data is clean")

In [0]:
# Split into wind and solar, then join on timestamp
# This merges the two row types into one clean row per hour

# Step 1: Separate wind rows (rows that came from the wind tab)
wind_df = silver_df.filter(col("wind_gen_mw").isNotNull())\
    .select(
        "timestamp","year","month","day","hour_of_day","ercot_load_mw","wind_gen_mw",
        "wind_capacity_factor","wind_pct_of_load","wind_1hr_mw_change","total_wind_installed_mw"
    )

# Step 2: Separate solar rows (rows that came from the solar tabl)
solar_df = silver_df.filter(col("solar_gen_mw").isNotNull())\
    .select(
        "timestamp","solar_gen_mw","solar_capacity_factor","solar_pct_of_load","solar_1hr_mw_change",
        "total_solar_installed_mw","daytime_hour","ramping_daytime_hour"
    )

# Step 3: Fix solar nighttime values (1->0) now that rows are separated
solar_df = solar_df.withColumn("solar_gen_mw",
                               when(
                                   (col("solar_gen_mw") == 1) & (col("daytime_hour") == False),
                                   lit(0)).otherwise(col("solar_gen_mw"))
                               )

print(f"Wind rows: {wind_df.count()}")
print(f"Solar rows: {solar_df.count()}")
wind_df.show(3)
solar_df.show(3)

In [0]:
# Using inner join to only keep timestamps that exists in BOTH tables. If a timestamp exists in wind but not solar, it gets dropped because every hour should have both wind and solar data

silver_joined_df = wind_df.join(
    solar_df,
    on = "timestamp",
    how = "inner"
)

# Verify the join results
print(f"Wind rows: {wind_df.count()}")
print(f"Solar rows: {solar_df.count()}")
print(f"Joined rows: {silver_joined_df.count()}")
print(f"Columns: {len(silver_joined_df.columns)}")

# Quick sanity check (show one daytime and one nighttime row)
print("\nDaytime example:")
silver_joined_df.filter(
    (col("hour_of_day") == 12) & (col("year") == 2024)
).show(2)

print("\nNighttime example:")
silver_joined_df.filter(
    (col("hour_of_day") == 2) & (col("year") == 2024)
).show(2)

In [0]:
# check wind_df for duplicates
print("==== Wind duplicates ====")
wind_df.groupBy("timestamp").agg(count("*").alias("row_count"))\
    .filter(col("row_count") > 1).orderBy("timestamp").show(10, truncate=False)

# check solar_df for duplicates
print("==== Solar duplicates ====")
solar_df.groupBy("timestamp").agg(count("*").alias("row_count"))\
    .filter(col("row_count") > 1).orderBy("timestamp").show(10, truncate=False)

In [0]:
print("==== Wind DST rows ====")
wind_df.filter(col("timestamp") == "2024-11-03 01:00:00").show(truncate=False)

print("==== Solar DST rows ====")
solar_df.filter(col("timestamp") == "2024-11-03 01:00:00").show(truncate=False)

In [0]:
# Step 1 — Add occurrence flag (same as before)
wind_window = Window.partitionBy("timestamp") \
                    .orderBy(col("wind_1hr_mw_change").desc())

solar_window = Window.partitionBy("timestamp") \
                     .orderBy(col("solar_gen_mw").desc())

wind_df_flagged = wind_df.withColumn(
    "dst_occurrence",
    row_number().over(wind_window)
)

solar_df_final = solar_df.withColumn(
    "dst_occurrence",
    row_number().over(solar_window)
).filter(col("dst_occurrence") == 1) \
 .drop("dst_occurrence")

# Step 2 — DST transition dates (fall-back Sunday each year)
# These are the specific dates clocks fall back in Central Time
dst_fall_back_dates = ["2023-11-05", "2024-11-03", "2025-11-02"]

# Step 3 — Correct offset logic
wind_df_with_offset = wind_df_flagged.withColumn(
    "date_str",
    F.date_format(col("timestamp"), "yyyy-MM-dd")
).withColumn(
    "utc_offset_hours",
    when(
        # Ambiguous DST hour occurrence 1 = CDT = UTC-5
        (col("date_str").isin(dst_fall_back_dates)) &
        (col("hour_of_day") == 1) &
        (col("dst_occurrence") == 1),
        F.lit(5)
    ).when(
        # Ambiguous DST hour occurrence 2 = CST = UTC-6
        (col("date_str").isin(dst_fall_back_dates)) &
        (col("hour_of_day") == 1) &
        (col("dst_occurrence") == 2),
        F.lit(6)
    ).when(
        # On DST fall-back day, hours >= 2 are all CST = UTC-6
        (col("date_str").isin(dst_fall_back_dates)) &
        (col("hour_of_day") >= 2),
        F.lit(6)
    ).when(
        # Standard CDT months (mid-March to early November) = UTC-5
        (col("month") >= 3) & (col("month") <= 10),
        F.lit(5)
    ).otherwise(
        # Standard CST months (November to mid-March) = UTC-6
        F.lit(6)
    )
).drop("date_str")

# Step 4 — Apply offset to get UTC timestamp
wind_df_final = wind_df_with_offset.withColumn(
    "timestamp_utc",
    (col("timestamp").cast("long") +
     col("utc_offset_hours") * 3600
    ).cast("timestamp")
).drop("utc_offset_hours")

# Step 5 — Verify DST rows
print("=== Wind DST rows after fix ===")
wind_df_final.filter(
    (col("year") == 2024) &
    (col("month") == 11) &
    (col("day") == 3) &
    (col("hour_of_day").isin([1, 2]))
).select(
    "timestamp", "timestamp_utc",
    "dst_occurrence", "hour_of_day", "wind_gen_mw"
).orderBy("timestamp_utc").show(10, truncate=False)

# Step 6 — Final duplicate check
print("=== Final UTC duplicate check ===")
dupes = wind_df_final.groupBy("timestamp_utc") \
    .agg(count("*").alias("row_count")) \
    .filter(col("row_count") > 1)

dupe_count = dupes.count()
if dupe_count == 0:
    print("No duplicate UTC timestamps — clean!")
else:
    print(f"WARNING: {dupe_count} duplicates remain")
    dupes.show(truncate=False)

print(f"\nWind rows: {wind_df_final.count()}")
print(f"Solar rows: {solar_df_final.count()}")

In [0]:
# Inspect all remaining duplicates
print("=== March duplicate ===")
wind_df_final.filter(
    col("timestamp_utc") == "2024-03-01 05:00:00"
).select(
    "timestamp", "timestamp_utc",
    "dst_occurrence", "month", 
    "day", "hour_of_day", "wind_gen_mw"
).show(truncate=False)

print("=== November duplicate ===")
wind_df_final.filter(
    col("timestamp_utc") == "2024-11-03 06:00:00"
).select(
    "timestamp", "timestamp_utc",
    "dst_occurrence", "month",
    "day", "hour_of_day", "wind_gen_mw"
).show(truncate=False)

In [0]:
# For the 6 remaining duplicate UTC timestamps
# keep the row with higher wind generation (first occurrence)
# Document: DST transition hours retain one observation only
dedup_window = Window.partitionBy("timestamp_utc") \
                     .orderBy(col("wind_gen_mw").desc())

wind_df_final_clean = wind_df_final.withColumn(
    "utc_rank",
    row_number().over(dedup_window)
).filter(col("utc_rank") == 1) \
 .drop("utc_rank", "dst_occurrence")

# Verify
dupes_remaining = wind_df_final_clean \
    .groupBy("timestamp_utc") \
    .agg(count("*").alias("row_count")) \
    .filter(col("row_count") > 1) \
    .count()

print(f"Remaining duplicates: {dupes_remaining}")
print(f"Final wind rows: {wind_df_final_clean.count()}")

In [0]:
from pyspark.sql.functions import month as spark_month

# Solar UTC conversion using timestamp directly for month extraction
solar_df_with_offset = solar_df_final.withColumn(
    "month_num",
    spark_month(col("timestamp"))
).withColumn(
    "utc_offset_hours",
    when(
        (col("month_num") >= 3) & (col("month_num") <= 10),
        F.lit(5)
    ).otherwise(F.lit(6))
).drop("month_num")

solar_df_utc = solar_df_with_offset.withColumn(
    "timestamp_utc",
    (col("timestamp").cast("long") +
     col("utc_offset_hours") * 3600
    ).cast("timestamp")
).drop("utc_offset_hours")

# Dedup solar on UTC timestamp
solar_dedup_window = Window.partitionBy("timestamp_utc") \
                           .orderBy(col("solar_gen_mw").desc())

solar_df_final_clean = solar_df_utc.withColumn(
    "utc_rank",
    row_number().over(solar_dedup_window)
).filter(col("utc_rank") == 1) \
 .drop("utc_rank")

print(f"Solar rows: {solar_df_final_clean.count()}")

solar_dupes = solar_df_final_clean \
    .groupBy("timestamp_utc") \
    .agg(count("*").alias("row_count")) \
    .filter(col("row_count") > 1) \
    .count()
print(f"Solar duplicates: {solar_dupes}")

In [0]:
# Drop duplicate timestamp column from solar before joining
solar_df_final_clean = solar_df_final_clean.drop("timestamp")

# Join on UTC timestamp — both tables now have clean, unique UTC timestamps
silver_joined_df = wind_df_final_clean.join(
    solar_df_final_clean,
    on="timestamp_utc",
    how="inner"
)

print(f"Joined rows: {silver_joined_df.count()}")
print(f"Columns: {len(silver_joined_df.columns)}")
silver_joined_df.show(3, truncate=False)

In [0]:
# Write Silver layer
silver_joined_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.medallion_lakehouse.silver_generation")

print("Silver layer written successfully")

# Final verification
spark.sql("""
    SELECT
        COUNT(*) as total_rows,
        MIN(timestamp_utc) as earliest_utc,
        MAX(timestamp_utc) as latest_utc,
        COUNT(DISTINCT year) as years_covered,
        SUM(CASE WHEN wind_gen_mw IS NULL THEN 1 ELSE 0 END) as wind_nulls,
        SUM(CASE WHEN solar_gen_mw IS NULL THEN 1 ELSE 0 END) as solar_nulls
    FROM workspace.medallion_lakehouse.silver_generation
""").show(truncate=False)